In [4]:
from feat_engin_helpers import *

KeyboardInterrupt: 

In [ ]:
Y, cases, unit_ids = load_uc()
long = to_long(Y, cases, unit_ids)
sub = long[long["year"] <= 2021]

features = []

#2. Attach raw features
price = read_timeseries("../sintef_project/data/kernel/Historical_day_ahead_price_2015_2025.csv")["Day-ahead price (EUR/MWh)"]
attach_hourly(long, price, "price")
features.append("price")

inflow = read_timeseries("../sintef_project/data/kernel/Historical_inflow_1958_2025.csv")
inflow = inflow.loc['2015-01-01':]
for res in inflow:
    attach_hourly(long, inflow[res], "inflow_"+res)
    features.append("inflow_"+res)

storage = read_timeseries("../sintef_project/data/kernel/Historical_volume_2015_2024.csv")
for res in storage:
    if res == "Hyljelihyl":
        continue
    attach_daily(long, storage[res], "storage"+res)
    features.append("storage"+res)

value = read_timeseries("../sintef_project/data/kernel/Synthetic_water_value_2015_2024.csv")
for res in value:
    attach_daily(long, value[res], "value_"+res)
    features.append("value_"+res)

min_vol = read_timeseries("../sintef_project/data/extended/Constraint_min_volume.csv")
for res in min_vol:
    attach_hourly(long, min_vol[res], "minVolume_"+res)
    features.append("minVolume_"+res)

min_flow = read_timeseries("../sintef_project/data/extended/Constraint_min_flow.csv")
for res in min_flow:
    attach_hourly(long, min_flow[res], "minFlow_"+res)
    features.append("minFlow_"+res)

# TEST
train = sub[sub["case_idx"] % 7 == 0]
all_scores = []

for unit, g in train.groupby("unit", observed=True):
    unit_scores = filter_scores(g, features, "y")
    all_scores.append(unit_scores)

    scores = pd.concat(all_scores, ignore_index=True)
    scores = scores.sort_values(by="strength", ascending=False).reset_index(drop=True)
print(scores)

In [14]:
# Rolling prices
long = to_long(Y, cases, unit_ids)

features = []

price_rolling_day = price.rolling(24, min_periods=1).sum()
price_rolling_week = price.rolling(24 * 7, min_periods=1).sum()
attach_hourly(long, price_rolling_day, "rolling_prices_day")
attach_hourly(long, price_rolling_week, "rolling_prices_week")
features.extend(["rolling_prices_day", "rolling_prices_week"])

# TEST
train = sub[sub["case_idx"] % 7 == 0]

for unit, g in train.groupby("unit", observed=True):
    unit_scores = filter_scores(g, features, "y")
    all_scores.append(unit_scores)

    scores = pd.concat(all_scores, ignore_index=True)
    scores = scores.sort_values(by="strength", ascending=False).reset_index(drop=True)
print(scores)

                 feature  correlation       auc  strength  sign_consistency
0    value_Botnedalsvatn    -0.613364  0.143637  0.356363          1.000000
1        value_Byrtevatn    -0.453004  0.188299  0.311701          1.000000
2        value_Kjelavatn    -0.537660  0.189900  0.310100          1.000000
3         value_Staavatn    -0.537660  0.189900  0.310100          1.000000
4      value_Bitdalsvatn    -0.524308  0.196017  0.303983          1.000000
..                   ...          ...       ...       ...               ...
891  minVolume_Byrtevatn     0.000950  0.500338  0.000338          0.428571
892      storageLangesae    -0.000234  0.499835  0.000165          0.714286
893        storageBandak    -0.000089  0.499948  0.000052          0.571429
894     minFlow_b_Bandak     0.000094  0.500049  0.000049          0.571429
895     minFlow_b_Bandak    -0.000068  0.499967  0.000033          0.571429

[896 rows x 5 columns]
